# 01 · Classical Baseline Segmenter — Component 3

**Phase 1, Step 1.1.** Runs `segmentation/classical.py` on **train-split images only** and checks it
against Konan. Two methods are compared:

| Method | Seeds | Cell size decided by |
|--------|-------|----------------------|
| `ridge` (default, v2) | basins enclosed by dark walls (Sato ridge filter + h-minima) | the image (wall contrast) |
| `seeds` (v1) | bright cell centres with a minimum spacing | the spacing parameter — v1 merged cells (area ×1.6) |

| # | Section | Output (in `outputs/classical/`) |
|---|---------|----------------------------------|
| 1 | Example overlays: ridge vs seeds | `01_examples.png` |
| 2 | Batch run on train images, both methods | `02_batch_results.csv` |
| 3 | Comparison with Konan | `03_vs_konan.png` |
| 4 | Sensitivity: cell-size assumption and h_threshold | `04_sensitivity.png` |

ECD uses the **preliminary** scale (0.752 µm/px) — a sanity check, not a reported result.
A CPU runtime is enough (≈ 8–12 min in total).

## 0 · Setup

In [ ]:
# Mount Drive, pull the latest code FIRST, then run the shared setup script.
from google.colab import drive; drive.mount("/content/drive")
REPO_DIR = "/content/drive/MyDrive/Colab Notebooks/Research_Project/xai-cornea-grading"
!git -C "{REPO_DIR}" pull --ff-only
%run "{REPO_DIR}/ml/cellular/colab_setup.py"

In [ ]:
from dataclasses import replace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from scipy import stats
from tqdm.auto import tqdm

from config import load_config
from segmentation.classical import ClassicalParams, segment, overlay

cfg = load_config()
P = cfg["paths"]
OUT = P["outputs_dir"] / "classical"
OUT.mkdir(parents=True, exist_ok=True)
SEED = cfg["split"]["seed"]
UM_PER_PX = cfg["scale"]["um_per_px_prelim"]

params = ClassicalParams.from_config(cfg)                 # method from config.yaml ("ridge")
METHODS = {"ridge": replace(params, method="ridge"), "seeds": replace(params, method="seeds")}
EXPECTED_AREA = params.expected_area_px
sns.set_theme(style="whitegrid", context="notebook")

man = pd.read_csv(P["splits_dir"] / "manifest.csv")
train = man[man["split"] == "train"].copy()               # never touch val / test here
print(f"Train images: {len(train)}")
print(f"Expected hexagon area at {params.cell_px} px spacing: {EXPECTED_AREA:.0f} px "
      f"(Konan median {train['mean_area'].median():.0f} µm² ≈ {train['mean_area'].median() / UM_PER_PX**2:.0f} px)")


def load_gray(image_file):
    return cv2.imread(str(P["images_dir"] / image_file), cv2.IMREAD_GRAYSCALE)


def save(fig, name):
    fig.savefig(OUT / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.show()


def zoom_window(labels, shape, size=300):
    ys, xs = np.nonzero(labels)
    cy, cx = (int(np.median(ys)), int(np.median(xs))) if len(ys) else (shape[0] // 2, shape[1] // 2)
    return (int(np.clip(cy - size // 2, 0, shape[0] - size)), int(np.clip(cx - size // 2, 0, shape[1] - size)))

## 1 · Example overlays — ridge (v2) vs seeds (v1)
Green = kept complete cells, red = walls, darkened = too dark to analyse. Same zoom window for both.
Check: does each red outline enclose **one** real cell? Are walls on the dark lines?

In [ ]:
examples = train.groupby("ecd_band").sample(n=2, random_state=SEED)   # 2 per ECD band

fig, axes = plt.subplots(len(examples), 3, figsize=(20, 5.6 * len(examples)))
for row_axes, (_, r) in zip(axes, examples.iterrows()):
    img = load_gray(r["image_file"])
    res = {m: segment(img, p) for m, p in METHODS.items()}
    y0, x0 = zoom_window(res["ridge"].labels, img.shape)
    ov = {m: overlay(img, rr) for m, rr in res.items()}

    row_axes[0].imshow(ov["ridge"])
    row_axes[0].add_patch(plt.Rectangle((x0, y0), 300, 300, fill=False, ec="yellow", lw=2))
    row_axes[0].set_title(f"{r['image_id']} | Konan ECD {r['ecd']:.0f}\n"
                          f"ridge: {res['ridge'].n_cells} cells, ECD {res['ridge'].ecd(UM_PER_PX):.0f}", fontsize=10)
    for ax, m in zip(row_axes[1:], ["ridge", "seeds"]):
        ax.imshow(ov[m][y0:y0 + 300, x0:x0 + 300])
        ax.set_title(f"zoom · {m}: mean area {res[m].mean_area_px:.0f} px "
                     f"(expected ≈ {EXPECTED_AREA:.0f}), kept {res[m].kept_fraction:.0%}", fontsize=10)
    for ax in row_axes:
        ax.axis("off")
save(fig, "01_examples")

## 2 · Batch run on train images — both methods
150 random train images (≈ 5–8 min). Per image and method: complete cells, mean area, CV, ECD
(preliminary scale) and the share of watershed regions kept.

In [ ]:
N_BATCH = 150
batch = train.sample(min(N_BATCH, len(train)), random_state=SEED)

rows = []
for _, r in tqdm(batch.iterrows(), total=len(batch), desc="Segmenting"):
    img = load_gray(r["image_file"])
    for m, p in METHODS.items():
        res = segment(img, p)
        rows.append({
            "image_id": r["image_id"], "method": m,
            "n_cells": res.n_cells, "kept_fraction": res.kept_fraction,
            "mean_area_px": res.mean_area_px, "mean_area_um2": res.mean_area_px * UM_PER_PX ** 2,
            "cv": res.cv_percent(), "ecd": res.ecd(UM_PER_PX),
            "konan_num_cells": r["num_cells"], "konan_mean_area": r["mean_area"],
            "konan_cv": r["cv"], "konan_ecd": r["ecd"],
        })
results = pd.DataFrame(rows)
results.to_csv(OUT / "02_batch_results.csv", index=False)
display(results.groupby("method")[["n_cells", "kept_fraction", "mean_area_px", "cv", "ecd"]]
               .median().round(2))

## 3 · Comparison with Konan
* **Mean area / ECD**: a *constant* ratio means a scale offset (fixed in Phase 4); a near-zero
  correlation means the method does not follow the real cell size.
* **CV**: sensitive to over-/under-segmentation. Konan measures ~110 hand-picked cells, we measure
  every complete cell in the valid area, so some difference is expected.

In [ ]:
def pct_err(a, b):
    return 100 * (a - b).abs() / b

PAIRS = [("mean_area_um2", "konan_mean_area", "Mean cell area (µm²)"),
         ("ecd", "konan_ecd", "ECD (cells/mm²)"),
         ("cv", "konan_cv", "CV (%)")]

fig, axes = plt.subplots(len(METHODS), 3, figsize=(18, 5 * len(METHODS)))
summary = {}
for row_axes, m in zip(axes, METHODS):
    d_m = results[results["method"] == m]
    for ax, (ours, konan, label) in zip(row_axes, PAIRS):
        d = d_m[[ours, konan]].dropna()
        ax.scatter(d[konan], d[ours], s=10, alpha=0.6)
        lims = [min(d.min()), max(d.max())]
        ax.plot(lims, lims, "r--", lw=1, label="y = x")
        rho, _ = stats.spearmanr(d[konan], d[ours])
        ax.set(xlabel=f"Konan {label}", ylabel=f"{m} {label}",
               title=f"{m} · {label}\nSpearman ρ = {rho:.2f}, median |err| = {pct_err(d[ours], d[konan]).median():.1f}%")
        ax.legend()
    summary[m] = {
        "median_cells_per_image": d_m["n_cells"].median(),
        "median_kept_fraction": d_m["kept_fraction"].median(),
        "area_ratio_median (ours/konan)": (d_m["mean_area_um2"] / d_m["konan_mean_area"]).median(),
        "area_spearman": stats.spearmanr(d_m["konan_mean_area"], d_m["mean_area_um2"])[0],
        "ecd_median_abs_pct_err": pct_err(d_m["ecd"], d_m["konan_ecd"]).median(),
        "cv_median_abs_pct_err": pct_err(d_m["cv"], d_m["konan_cv"]).median(),
    }
save(fig, "03_vs_konan")
display(pd.DataFrame(summary).round(3))

## 4 · Sensitivity (ridge method)
* **cell_px ×0.85 / ×1.15** — for `ridge` this only moves the size filter, so mean area should change
  **much less** than for `seeds` (v1: 775 → 1061 → 1400 px). A flat line = data-driven.
* **h_threshold** — the one real knob: too low over-splits cells, too high merges them.

In [ ]:
sample = batch.head(30)
ridge = METHODS["ridge"]
variants = ([("cell_px", f, replace(ridge, cell_px=ridge.cell_px * f)) for f in (0.85, 1.0, 1.15)] +
            [("h_threshold", h, replace(ridge, h_threshold=h)) for h in (0.06, 0.12, 0.20)])

sens = []
for _, r in tqdm(sample.iterrows(), total=len(sample), desc="Sensitivity"):
    img = load_gray(r["image_file"])
    for knob, value, p in variants:
        res = segment(img, p)
        sens.append({"knob": knob, "value": value, "image_id": r["image_id"],
                     "mean_area_px": res.mean_area_px, "n_cells": res.n_cells,
                     "konan_area_px": r["mean_area"] / UM_PER_PX ** 2})
sens = pd.DataFrame(sens)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, knob in zip(axes, ["cell_px", "h_threshold"]):
    d = sens[sens["knob"] == knob]
    sns.boxplot(data=d, x="value", y="mean_area_px", ax=ax)
    ax.axhline(d["konan_area_px"].median(), color="r", ls="--", label="Konan median (prelim. scale)")
    ax.set_title(f"ridge · mean cell area vs {knob}"); ax.legend()
save(fig, "04_sensitivity")
display(sens.groupby(["knob", "value"])[["mean_area_px", "n_cells"]].median().round(1))

## Notes (fill in after running)
* Visual quality, ridge vs seeds (Section 1): …
* Agreement with Konan (Section 3): …
* Sensitivity (Section 4): …
* Typical failure modes (to fix during annotation): …